<a href="https://colab.research.google.com/github/author-sanjay/AirSafetyAI/blob/Data-Normalization/AirCraftAccidentDataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from google.colab import drive
from collections import defaultdict
import json
import pandas as pd
import re
from datetime import datetime

# Data Collection

###### Data collection has been done already from airsafety db from year 2000 to 2025 resulting up to 6500+ recorded incidents found to train the model. Please note that this data is only being used for research purposes and model training

# Data Processing

#### Finding and Deleting Duplicates in dat

In [ ]:


file_path = "/content/drive/MyDrive/accidents.json"

# Load your data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Dictionary to track occurrences
seen = defaultdict(list)

for idx, entry in enumerate(data):
    # Composite key: Date + Time + Registration + Location
    key = f"{entry.get('Date','')}_{entry.get('Time','')}_{entry.get('Registration','')}_{entry.get('Location','')}"
    seen[key].append(idx)

# Find duplicates (keys with more than 1 entry)
duplicates = {k: v for k, v in seen.items() if len(v) > 1}

print(f"✅ Total entries: {len(data)}")
print(f"⚠️ Potential duplicates found: {len(duplicates)}")

✅ Total entries: 6791
⚠️ Potential duplicates found: 0


#### Data Normalisation

In [ ]:


file_path = "/content/drive/MyDrive/accidents.json"
cleaned_file_path = "/content/drive/MyDrive/accidents_cleaned.json"
csv_file_path = "/content/drive/MyDrive/accidents_cleaned.csv"

# --- Load JSON into pandas ---
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)


###### Fixing Unknown Dates

In [ ]:


# --- Handle unk. date YYYY ---
def normalize_date(val):
    if pd.isna(val):
        return None
    val = str(val).strip()

    # If format is "unk. date YYYY"
    match = re.match(r"unk\. date (\d{4})", val, flags=re.IGNORECASE)
    if match:
        year = int(match.group(1))
        return f"{year}-12-31"

    # Try normal parsing
    try:
        return pd.to_datetime(val, errors="coerce").strftime("%Y-%m-%d")
    except Exception:
        return None

df["Date"] = df["Date"].apply(normalize_date)
print(df.head())


         Date      Time                            Type  \
0  2000-01-01  13:00 LT          Cessna 550 Citation II   
1  2000-01-03             Beechcraft 200 Super King Air   
2  2000-01-04  17:25 LT  Beechcraft B200 Super King Air   
3  2000-01-05     13:25  Embraer EMB-110P1A Bandeirante   
4  2000-01-07                             Antonov An-26   

                    Owner/operator Registration       MSN Total airframe hrs  \
0               US Customs Service       N752CC  550-0018        12159 hours   
1  Kalahari Air Services & Charter       A2-AEZ    BB-421                NaN   
2                          Private       N895TT   BB-1239         3238 hours   
3         Skypower Express Airways       5N-AXL    110455                NaN   
4                          Unknown       D2-FBR      7206                NaN   

                     Engine model                     Fatalities  \
0                     P&W JT15D-4   Fatalities: 0 / Occupants: 3   
1                           